In [16]:
uv pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.7 environment at: c:\Users\ddddd\AppData\Local\Programs\Python\Python313
Audited 1 package in 3ms


In [2]:
import re
import polars as pl
import plotly.express as px

df = pl.read_parquet("spy_10k_2015_present.parquet")

# Baseline ESG keywords/phrases
keywords = ["environment", "sustainability", "climate change", "carbon emissions"]

# Build regex patterns:
# - case-insensitive
# - word boundaries for single words
# - allow whitespace/hyphen between words in phrases (e.g., "climate-change")
patterns = {}
for k in keywords:
    if " " in k:
        parts = [re.escape(p) for p in k.split()]
        pat = r"(?i)\b" + r"[-\s]+".join(parts) + r"\b"
    else:
        pat = r"(?i)\b" + re.escape(k) + r"\b"
    patterns[k] = pat

wordcount_df = df.with_columns(
    [pl.col("text").str.count_matches(patterns[k]).alias(f"{k}_count") for k in keywords]
)

ticker = "AAPL"
section = "risk_factors"

plot_df = (
    wordcount_df
    .filter((pl.col("ticker") == ticker) & (pl.col("section") == section))
    .sort("filing_date")
    .select(["ticker", "filing_date"] + [f"{k}_count" for k in keywords])
    .to_pandas()
)

fig = px.line(
    plot_df,
    x="filing_date",
    y=[f"{k}_count" for k in keywords],
    title=f"Keyword Occurrences in 10-K Filings for {ticker} ({section})",
    labels={"filing_date": "Filing Date", "value": "Count"},
    markers=True,
)
fig.update_layout(legend_title_text="Keywords", template="plotly_white")
fig.update_yaxes(
    tickmode="linear",
    tick0=0,
    dtick=1
)

fig.show()


In [9]:
import polars as pl
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

YEAR = 2015

# Load only needed columns
df = pl.read_parquet(
    "spy_10k_2015_present.parquet",
    columns=["cik", "gics_sector", "filing_period"],
)

# Aggregate unique firms per sector
sector_2015 = (
    df.with_columns(
        pl.col("cik").cast(pl.Utf8),
        pl.col("gics_sector").cast(pl.Utf8),
        pl.col("filing_period").cast(pl.Date),
        pl.col("filing_period").dt.year().alias("year"),
    )
    .filter(
        (pl.col("year") == YEAR)
        & pl.col("gics_sector").is_not_null()
        & pl.col("cik").is_not_null()
    )
    .group_by("gics_sector")
    .agg(pl.col("cik").n_unique().alias("n_firms"))
    .sort("n_firms", descending=True)
)

# Convert to pandas for Plotly
plot_df = sector_2015.to_pandas()

# Simple bar chart (THIS is what you want)
fig = px.bar(
    plot_df,
    x="gics_sector",
    y="n_firms",
    title=f"Number of Companies by GICS Sector in {YEAR}",
    labels={
        "gics_sector": "GICS Sector",
        "n_firms": "Number of Companies",
    },
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-30,  # readable labels
)

fig.show()


In [ ]:
import re
import polars as pl
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

# --------------------
# Config
# --------------------
TICKER = "AAPL"
SECTION = "risk_factors"
START_YEAR = 2015
END_YEAR = 2024

# ESG lexicon (regex patterns)
ESG_TERMS = {
    "Environmental": {
        "climate risk": r"(?i)\bclimate[-\s]+risk\b",
        "climate change": r"(?i)\bclimate[-\s]+change\b",
        "greenhouse gases": r"(?i)\bgreenhouse[-\s]+gas(es)?\b",
        "emissions": r"(?i)\bemission(s)?\b",
        "carbon": r"(?i)\bcarbon\b",
        "net zero": r"(?i)\bnet[-\s]+zero\b",
        "renewables": r"(?i)\brenewable(s)?\b",
        "energy efficiency": r"(?i)\benergy[-\s]+efficien(cy|t)\b",
        "recycling": r"(?i)\brecycling\b",
        "pollution": r"(?i)\bpollution\b",
    },
    "Social": {
        "diversity": r"(?i)\bdiversit(y|ies)\b",
        "inclusion": r"(?i)\binclusion\b",
        "human rights": r"(?i)\bhuman rights?\b",
        "workplace safety": r"(?i)\bwork(place)?[-\s]+safety\b",
        "labor practices": r"(?i)\blabor[-\s]+(practices?|relations?)\b",
        "pay equity": r"(?i)\bpay[-\s]+equity\b",
        "employee wellbeing": r"(?i)\bemploye(e|ment)[-\s]+wellbeing\b",
        "training": r"(?i)\btraining\b",
        "community engagement": r"(?i)\bcommunity[-\s]+(engagement|outreach)\b",
    },
    "Governance": {
        "board independence": r"(?i)\bboard[-\s]+independence\b",
        "executive compensation": r"(?i)\bexecutive[-\s]+compensation\b",
        "shareholder rights": r"(?i)\bshareholder[-\s]+rights?\b",
        "audit committee": r"(?i)\baudit[-\s]+committee\b",
        "internal controls": r"(?i)\binternal[-\s]+controls?\b",
        "whistleblower": r"(?i)\bwhistleblow(er|ing)\b",
        "anti corruption": r"(?i)\banti[-\s]+corruption\b",
        "anti bribery": r"(?i)\banti[-\s]+bribery\b",
        "corporate governance": r"(?i)\bcorporate[-\s]+governance\b",
    },
}

# --------------------
# Load & filter data (once)
# --------------------
df = pl.read_parquet(
    "spy_10k_2015_present.parquet",
    columns=["ticker", "filing_period", "section", "text"],
)

df = (
    df.with_columns(
        pl.col("filing_period").cast(pl.Date),
        pl.col("filing_period").dt.year().alias("year"),
    )
    .filter(
        (pl.col("ticker") == TICKER)
        & (pl.col("section") == SECTION)
        & (pl.col("year") >= START_YEAR)
        & (pl.col("year") <= END_YEAR)
        & pl.col("text").is_not_null()
    )
)

# --------------------
# Loop over E / S / G and plot
# --------------------
for pillar, terms in ESG_TERMS.items():

    tmp = df.with_columns(
        [
            pl.col("text").str.count_matches(pattern).alias(term)
            for term, pattern in terms.items()
        ]
    )

    plot_df = (
        tmp.group_by("year")
           .agg([pl.sum(term).alias(term) for term in terms.keys()])
           .sort("year")
           .melt(id_vars="year", variable_name="term", value_name="count")
           .to_pandas()
    )

    fig = px.line(
        plot_df,
        x="year",
        y="count",
        color="term",
        markers=True,
        title=f"{pillar} Keyword Occurrences in AAPL 10-K Risk Factors (2015–2024)",
        labels={
            "year": "Filing Year",
            "count": "Keyword Occurrences",
            "term": f"{pillar} Term",
        },
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
    )

    fig.update_xaxes(tickmode="linear", tick0=START_YEAR, dtick=1)
    fig.update_yaxes(tickmode="linear", tick0=0)

    fig.show()


C:\Users\ddddd\AppData\Local\Temp\ipykernel_21348\976313252.py:68: DeprecationWarning:

`DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`

